In [ ]:
import numpy as np
import pandas as pd


try:
    from kaggle_evaluation.core.base import ParticipantVisibleError
except Exception:
    class ParticipantVisibleError(Exception):
        """Fallback for local testing outside Kaggle."""
        pass


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = "image_id") -> float:
    """
    Compute the official leaderboard score for the UMUD Muscle Architecture in 
    Ultrasound Data challenge.

    This metric is intended for predictions of three
    continuous targets from ultrasound images:

    - ``pa_deg`` : pennation angle in degrees
    - ``fl_mm``  : fascicle length in millimetres
    - ``mt_mm``  : muscle thickness in millimetres

    The function returns one scalar score because Kaggle ranks submissions using
    a single numeric value. Internally, the score is defined as a weighted mean
    of normalized mean absolute errors (MAE) across the three targets, with
    normalized median absolute error (MedAE) and normalized root mean squared
    error (RMSE) embedded as deterministic tie-breakers.

    Lower values are better.

    Parameters
    ----------
    solution : pandas.DataFrame
        Ground-truth table supplied by Kaggle. Must contain the row ID column and
        the target columns ``pa_deg``, ``fl_mm``, and ``mt_mm``. It may also
        contain ``subject_id`` if subject-level aggregation is enabled.
    submission : pandas.DataFrame
        Participant prediction table. Must contain the same row ID column and the
        required prediction columns ``pa_deg``, ``fl_mm``, and ``mt_mm``.
    row_id_column_name : str
        Name of the identifier column used to align rows between ``solution`` and
        ``submission``.

    Returns
    -------
    float
        Official leaderboard score. Lower is better.

    Raises
    ------
    ParticipantVisibleError
        Raised when the submission is malformed, such as missing columns,
        duplicate IDs, missing rows, non-numeric predictions, NaNs, or infinite
        values.

    Notes
    -----
    **Primary metric**

    For each target c in {PA, FL, MT}, let MAE_c be the mean absolute error and
    tau_c the predefined tolerance for that target. The primary score is

    .. math::

        S_1 = \\frac{1}{\\sum_c w_c}
              \\sum_c w_c \\frac{MAE_c}{\\tau_c}

    where w_c are target weights.

    **Tie-breaking**

    Kaggle only accepts one returned score, so deterministic tie-breaking is
    encoded directly into that scalar:

    .. math::

        S = S_1 + \\epsilon_1 S_2 + \\epsilon_2 S_3

    where S_2 is the weighted mean of normalized median absolute errors and S_3
    is the weighted mean of normalized RMSE values. The epsilon values are chosen
    to be very small so they do not materially alter the primary ranking unless
    two submissions are effectively tied on the primary score.

    **Evaluation unit**

    By default, each row is treated as one independently scored image. If
    ``eval_unit='subject'`` is enabled and ``subject_id`` is present in the
    ground-truth table, the evaluator first averages predictions and ground truth
    within each subject and then computes metrics across subjects. This can be
    useful when multiple images belong to the same participant and the organizers
    want each participant to contribute equally.


    Examples
    --------
    >>> import pandas as pd
    >>> solution = pd.DataFrame({
    ...     "image_id": [1, 2, 3],
    ...     "pa_deg": [10.0, 20.0, 30.0],
    ...     "fl_mm": [50.0, 60.0, 55.0],
    ...     "mt_mm": [15.0, 16.0, 17.0],
    ... })
    >>> submission = pd.DataFrame({
    ...     "image_id": [1, 2, 3],
    ...     "pa_deg": [11.0, 21.0, 29.0],
    ...     "fl_mm": [49.0, 62.0, 54.0],
    ...     "mt_mm": [15.2, 15.8, 16.5],
    ... })
    >>> float(score(solution, submission, "image_id")) >= 0
    True
    """
    # ------------------------------------------------------------------
    # Organizer settings
    # ------------------------------------------------------------------
    target_cols = ("pa_deg", "fl_mm", "mt_mm")

    # tolerances (based on literature MDCs) used for score normalization
    tolerances = {
        "pa_deg": 6.0,
        "fl_mm": 12.0,
        "mt_mm": 3.0,
    }

    weights = {
        "pa_deg": 1.0,
        "fl_mm": 1.0,
        "mt_mm": 1.0,
    }

    # "image"  -> each row contributes equally
    eval_unit = "image"

    # Tiny coefficients for deterministic tie-breaking. 
    # These are used in the secondary metric (MedAE) and the tertiary metric (RMSE)
    eps_secondary = 1e-6
    eps_tertiary = 1e-9

    # ------------------------------------------------------------------
    # Validation
    # ------------------------------------------------------------------
    if row_id_column_name not in solution.columns:
        raise ParticipantVisibleError(
            f"Solution is missing id column '{row_id_column_name}'."
        )
    if row_id_column_name not in submission.columns:
        raise ParticipantVisibleError(
            f"Submission must include id column '{row_id_column_name}'."
        )

    for col in target_cols:
        if col not in solution.columns:
            raise ParticipantVisibleError(
                f"Solution is missing ground-truth column '{col}'."
            )
        if col not in submission.columns:
            raise ParticipantVisibleError(
                f"Submission is missing required prediction column '{col}'."
            )

    if solution[row_id_column_name].duplicated().any():
        raise ParticipantVisibleError(
            f"Duplicate ids found in solution column '{row_id_column_name}'."
        )
    if submission[row_id_column_name].duplicated().any():
        raise ParticipantVisibleError(
            f"Duplicate ids found in submission column '{row_id_column_name}'."
        )

    merged = solution.merge(
        submission,
        on=row_id_column_name,
        how="inner",
        suffixes=("_true", "_pred"),
    )

    if len(merged) != len(solution):
        missing = len(solution) - len(merged)
        raise ParticipantVisibleError(
            f"Submission is missing {missing} required row ids."
        )

    for col in target_cols:
        pred_col = f"{col}_pred"
        merged[pred_col] = pd.to_numeric(merged[pred_col], errors="coerce")
        if merged[pred_col].isna().any():
            raise ParticipantVisibleError(
                f"Column '{col}' must be numeric and contain no NaN values."
            )
        if np.isinf(merged[pred_col].to_numpy()).any():
            raise ParticipantVisibleError(
                f"Column '{col}' contains infinite values."
            )

    eval_df = merged

    # ------------------------------------------------------------------
    # Metric helpers
    # ------------------------------------------------------------------
    def _mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
        return float(np.mean(np.abs(y_pred - y_true)))

    def _medae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
        return float(np.median(np.abs(y_pred - y_true)))

    def _rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
        return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

    # ------------------------------------------------------------------
    # Compute normalized composite
    # ------------------------------------------------------------------
    primary = 0.0
    secondary = 0.0
    tertiary = 0.0
    weight_sum = float(sum(weights[c] for c in target_cols))

    for c in target_cols:
        y_true = eval_df[f"{c}_true"].to_numpy(dtype=float)
        y_pred = eval_df[f"{c}_pred"].to_numpy(dtype=float)

        tau = float(tolerances[c])
        w = float(weights[c])

        if not np.isfinite(tau) or tau <= 0:
            raise ParticipantVisibleError(
                f"Tolerance for '{c}' must be a positive finite number."
            )

        primary += w * (_mae(y_true, y_pred) / tau)
        secondary += w * (_medae(y_true, y_pred) / tau)
        tertiary += w * (_rmse(y_true, y_pred) / tau)

    primary /= weight_sum
    secondary /= weight_sum
    tertiary /= weight_sum

    final_score = primary + eps_secondary * secondary + eps_tertiary * tertiary
    return float(final_score)


# ----------------------------------------------------------------------
# Optional local helper for debugging / reporting outside Kaggle ranking
# ----------------------------------------------------------------------
def local_metric_report(solution: pd.DataFrame,
                        submission: pd.DataFrame,
                        row_id_column_name: str) -> pd.DataFrame:
    """
    Return a simple per-target metric table for local inspection.

    This is not used by Kaggle for ranking. It is only a convenience helper for
    organizer-side testing and documentation.
    """
    target_cols = ("pa_deg", "fl_mm", "mt_mm")

    merged = solution.merge(
        submission,
        on=row_id_column_name,
        how="inner",
        suffixes=("_true", "_pred"),
    )

    rows = []
    for c in target_cols:
        y_true = merged[f"{c}_true"].to_numpy(dtype=float)
        y_pred = merged[f"{c}_pred"].to_numpy(dtype=float)
        err = y_pred - y_true
        rows.append({
            "target": c,
            "mae": float(np.mean(np.abs(err))),
            "medae": float(np.median(np.abs(err))),
            "rmse": float(np.sqrt(np.mean(err ** 2))),
            "bias": float(np.mean(err)),
        })
    return pd.DataFrame(rows)


# Self-test
# solution_example = pd.DataFrame({
#    "image_id": [1, 2, 3, 4],
#    "pa_deg": [10.0, 20.0, 30.0, 25.0],
#    "fl_mm": [50.0, 60.0, 55.0, 70.0],
#    "mt_mm": [15.0, 16.0, 17.0, 18.0],
#})

# submission_example = pd.DataFrame({
#    "image_id": [1, 2, 3, 4],
#    "pa_deg": [11.0, 19.5, 29.0, 26.0],
#    "fl_mm": [49.0, 62.0, 54.0, 69.0],
#    "mt_mm": [15.2, 15.8, 16.5, 18.1],
#})

# print("Example score:", score(solution_example, submission_example, "image_id"))
# print(local_metric_report(solution_example, submission_example, "image_id"))